# Installation
The following are installation commands that I used to set up the virtual environment. All of this has already been done so all
switch to the virtual environment. 


In [2]:
%pip install -U pip setuptools wheel
%pip install -U spacy


  Using cached pip-26.1.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached wheel-0.47.0-py3-none-any.whl.metadata (2.3 kB)
Using cached pip-26.1.1-py3-none-any.whl (1.8 MB)
Using cached wheel-0.47.0-py3-none-any.whl (32 kB)

  Attempting uninstall: pip

    Found existing installation: pip 25.3

   -------------------- ------------------- 1/2 [pip]
    Uninstalling pip-25.3:
   -------------------- ------------------- 1/2 [pip]
      Successfully uninstalled pip-25.3
   -------------------- ------------------- 1/2 [pip]
   -------------------- ------------------- 1/2 [pip]
   -------------------- ------------------- 1/2 [pip]
   -------------------- ------------------- 1/2 [pip]
   -------------------- ------------------- 1/2 [pip]
   -------------------- ------------------- 1/2 [pip]
   -------------------- ------------------- 1/2 [pip]
   -------------------- ------------------- 1/2 [pip]
   -------------------- ------------------- 1/2 [pip]
   -------------------- -----------------

In [3]:
%pip list

Package                 Version
----------------------- -----------
annotated-doc           0.0.4
annotated-types         0.7.0
anyio                   4.13.0
asttokens               3.0.1
blis                    1.3.3
catalogue               2.0.10
certifi                 2026.4.22
charset-normalizer      3.4.7
click                   8.3.3
cloudpathlib            0.24.0
colorama                0.4.6
comm                    0.2.3
confection              1.3.3
cymem                   2.0.13
debugpy                 1.8.20
decorator               5.2.1
en_core_web_sm          3.8.0
executing               2.2.1
h11                     0.16.0
httpcore                1.0.9
httpx                   0.28.1
idna                    3.13
ipykernel               7.2.0
ipython                 9.13.0
ipython_pygments_lexers 1.1.1
jedi                    0.20.0
Jinja2                  3.1.6
jupyter_client          8.8.0
jupyter_core            5.9.1
markdown-it-py          4.2.0
MarkupSafe          

Switch to virtual enviroment using ./venv/scripts/Activate.bat in a vs code terminal
Then run: `python -m spacy download en_core_web_sm` in the terminal

In [4]:
import spacy
from spacy import displacy
from pathlib import Path
spacy.prefer_gpu()
nlp = spacy.load("en_core_web_sm")

In [5]:
'''
This function will take the text and render all the named enitites. 
It will then display it to the screen or save the render as an svg depending on the
"dipslya_in_jupyter" argument. 
'''
def renderEntities(text, output_name, display_in_jupyter):
    doc = nlp(text)
    file_name_display = output_name + "_display.svg" #Name of the output file
    # Which entities are we wanting to render. All the posible entities can be found at https://demos.explosion.ai/displacy-ent
    options = {"ents": ["LOC", "GPE"]} 
    
    if(display_in_jupyter):
        displacy.render(doc, style="ent", jupyter=True)
    else:

        svg = displacy.render(doc, style="ent", options=options, jupyter=false)
        output_path = Path("./outputs/" + file_name_display)
        output_path.open("w", encoding="utf-8").write(svg)




In [6]:
'''
This function will take the text and 
save all the entities found within the text as a json file.
'''
def extractEntities(text, output_name):
    doc = nlp(text)
    file_name_data = output_name + "_data.json"
    data = {}
    data['id'] = output_name
    data['entities']  = []
    for entity in doc.ents:
        '''
         The labels for the different types of name entities.
         LOC and GPE would likely have a lat / long. You can adjust which entities should be included similiar to the options in the
         renderEntities() function.
         The True is included just for testing and can be removed.
         '''
        if(entity.label_ == "LOC" or entity.label_ == "GPE" or True):
            data['entities'].append(entity.text)
        
    with open("./outputs/" + file_name_data, "w", encoding="utf-8") as data_file:
            json.dump(data, data_file, indent=1)

In [7]:
import json
def readJSON(dir, fileName):
    with open(dir + fileName, "r", encoding="utf-8") as file:
        return json.load(file)

In [8]:
filename =  "openalexworks.json" # This file contains the metadata, including the abstracts of the different works
dir = "../data/"
works = readJSON(dir, fileName=filename) # Read in the open alex works


In [9]:
#This is the first entry in the list. The abstract is accssed by the 'ab'.
# You can change the work by smiply changing the 0 to another number. 
work_abstract = works[0]['ab'] 

In [10]:
renderEntities(work_abstract, "test", True)
extractEntities(work_abstract, "test")

In [11]:
# Extract LOC/GPE entities from all works and write to a single output file.
# Keyed by OpenAlex work ID so downstream code can look up by ID directly.
all_entities = {}

for work in works:
    if 'ab' not in work or not work['ab']:
        continue

    doc = nlp(work['ab'])
    entities = [
        {
            "text": ent.text,
            "label": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char
        }
        for ent in doc.ents
        if ent.label_ in ("LOC", "GPE")
    ]

    all_entities[work['id']] = entities

output_path = "./outputs/all_entities.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_entities, f, indent=2)

print(f"Wrote {len(all_entities)} works to {output_path}")

Wrote 3132 works to ./outputs/all_entities.json
